# Object Builders

`cimbuilder.object_builder` has one builder class per CIM class: `ACLineSegmentBuilder` makes an
`ACLineSegment`, `BreakerBuilder` a `Breaker`, `EnergyConsumerBuilder` an `EnergyConsumer`, and so on. A
builder writes into a CIMantic Graphs model, one piece of the object at a time, and checks every attribute
and association name against the CIM profile as it goes. This page shows the pattern every builder
follows. [A feeder from scratch](3_2_feeder_from_scratch.ipynb) then puts it to work, and the later pages
cover each kind of equipment.

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import BaseVoltageBuilder, EnergyConsumerBuilder

## Start with a network

Every builder writes into a network, a CIMantic Graphs model, and a container, the `EquipmentContainer`
that new equipment goes in by default. `new_feeder_model` makes an empty `FeederModel` around a new
`Feeder` and adds a 60 Hz `BaseFrequency`. `authority` and `model` seed the UUID of every object, so the
same build gives the same UUIDs every time it runs.

In [2]:
network = new_feeder_model(
    authority="https://example.com/models",
    model="object_builders",
    Feeder="fdr_demo",
    name="demo",
)
container = network.container

print(type(network).__name__, "around", type(container).__name__, container.mRID)
print(f"base frequency: {network.find_by_mRID('base_freq').frequency.to('Hz'):g} Hz")

FeederModel around Feeder fdr_demo
base frequency: 60 Hz


To add equipment to an existing model instead, load it with CIMantic Graphs (for example a `FeederModel`
with an `XMLFile` connection, as the [substation builder](../02_substation_builder/2_1_substation_builder.ipynb)
pages do) and pass that model and its container to the builders.

## Create, then add one part at a time

A builder is constructed with the network and the container. `create()` makes the object and adds it to
the graph. It takes only the identity of the object: `mRID`, `name` and `description`.

In [3]:
BaseVoltageBuilder(network, container).create(
    mRID="bv_12_47kV", name="12.47 kV", nominalVoltage=Qty(12.47, "kV")
)

builder = EnergyConsumerBuilder(network, container)
load = builder.create(mRID="ld_671", name="671")
print(type(load).__name__, load.mRID, load.name)

EnergyConsumer ld_671 671


Each `add_*` method then fills in one CIM profile part, the slice of the class that one kind of model
exchange carries:

- `add_connectivity`, the connectivity part: Terminals and their nodes, the `EquipmentContainer`, the
  `BaseVoltage` and the per-phase children.
- `add_electrical`, the electrical part: ratings and steady-state impedances.
- `add_short_circuit`, the short-circuit part: zero- and negative-sequence data.
- `add_ssh`, the steady-state hypothesis: the operating point, such as `p` and `q`, a switch's `open`, a
  tap changer's `step`, and `inService`.
- `add_asset_info`, the asset info part: catalog data on an `AssetInfo`, such as a wire or inverter
  datasheet.

Some builders have more, such as `add_regulating_control` or `add_wire_configuration`. Call only the ones
the model needs. Each method takes the object plus keyword-only arguments, and returns the object plus
whatever it created:

In [4]:
load, terminal, phases = builder.add_connectivity(
    load, node="671", phaseConnection="Y", phases="ABC", BaseVoltage="bv_12_47kV"
)

print("terminal: ", terminal.mRID, "on node", terminal.ConnectivityNode.mRID)
print("phases:   ", [phase.mRID for phase in phases])
print("container:", load.EquipmentContainer.mRID)
print("voltage:  ", load.BaseVoltage.name)

terminal:  ld_671_T1 on node 671
phases:    ['ld_671_A', 'ld_671_B', 'ld_671_C']
container: fdr_demo
voltage:   12.47 kV


The Terminal and the per-phase `EnergyConsumerPhase` objects take their mRIDs from the load:
`<mRID>_T<sequence number>` and `<mRID>_<phase>`.

## Refer to other objects by mRID

Associations are mRID strings, such as `BaseVoltage="bv_12_47kV"`. The target has to exist already, so
build catalogs, base voltages and containers before the equipment that refers to them. A missing mRID
raises a `ValueError` rather than leaving the association empty:

In [5]:
other = builder.create(mRID="ld_692")
try:
    builder.add_electrical(other, LoadResponse="zip_missing")
except ValueError as error:
    print(error)

'ld_692': LoadResponse 'zip_missing' not found in graph (build it before the object that references it)


Nodes are the exception. A `node`, `node1` or `node2` that does not exist yet becomes a new
`ConnectivityNode` in the container, as `671` did above. Build the node first with
`ConnectivityNodeBuilder` to give it a name or a `BaseVoltage` (see
[Containers and nodes](3_3_containers_and_nodes.ipynb)).

## Values carry their units

Pass a `Qty` in any unit. It is converted and stored in CIM's SI unit, here W and VAr. Read a value back in
the unit you want with `.to()`. A plain number is taken to be in the SI unit already.

In [6]:
builder.add_ssh(load, p=Qty(1155, "kW"), q=Qty(660, "kVAr"))

print(load.p, "/", load.q)
print(f"{load.p.to('kW'):g} kW / {load.q.to('MVAr'):g} MVAr")

1155000.0 watt / 660000.0 VAr
1155 kW / 0.66 MVAr


## Enumerations are strings

CIM enumerations, such as `phaseConnection` or a switch's phase code, are passed as their string value and
stored as the CIM enumeration member. An unknown string raises.

In [7]:
print(repr(load.phaseConnection), repr(terminal.phases))

<PhaseShuntConnectionKind.Y: 'Y'> <PhaseCode.ABC: 'ABC'>


## Save the model

`cimgraph.utils.write_xml(network, "model.xml")` writes the whole graph as CIM RDF/XML, and
`write_json_ld` as JSON-LD. [A feeder from scratch](3_2_feeder_from_scratch.ipynb) writes a model and reads
it back.

## The builders

| Page | Builders |
|---|---|
| [A feeder from scratch](3_2_feeder_from_scratch.ipynb) | a source, transformer, breaker, line, load, capacitor and PV system in one model |
| [Containers and nodes](3_3_containers_and_nodes.ipynb) | `BaseVoltage`, `BasePower`, `Feeder`, `Substation`, `VoltageLevel`, `Bay`, `ConnectivityNode`, `BusbarSection`, `Junction`, `TopologicalNode` |
| [Lines](3_4_lines.ipynb) | `ACLineSegment`, `PerLengthSequenceImpedance`, `PerLengthPhaseImpedance`, wire and spacing catalogs, `SeriesCompensator` |
| [Switches](3_5_switches.ipynb) | `Breaker`, `Recloser`, `Fuse`, `Disconnector` and the other switch classes, `SwitchPhase`, `SwitchInfo` |
| [Transformers and regulators](3_6_transformers_and_regulators.ipynb) | `PowerTransformer`, `RatioTapChanger`, `TapChangerControl`, `VoltageRegulator` |
| [Transformer tanks](3_7_transformer_tanks.ipynb) | `TransformerTankConfiguration`, transformer banks built from tanks |
| [Loads and shunts](3_8_loads_and_shunts.ipynb) | `EnergyConsumer`, `LoadResponseCharacteristic`, `EnergyConnectionSchedule`, `LinearShuntCompensator`, `RegulatingControl` |
| [Sources and generation](3_9_sources_and_generation.ipynb) | `EnergySource`, `SynchronousMachine`, `AsynchronousMachine`, `PowerElectronicsConnection` with PV, battery and electric vehicle units |
| [Measurements and limits](3_10_measurements_and_limits.ipynb) | `Analog`, `Discrete`, `OperationalLimitSet` |